In [1]:
# 1. IMPORTS
import pandas as pd
import numpy as np
import os
import torch
from datasets import Dataset, Value
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, set_seed
)
from peft import get_peft_model, LoraConfig, TaskType
from sklearn.metrics import mean_squared_error, r2_score
import wandb

wandb.login()

SEED = 42
set_seed(SEED)

# 2. CONFIGURATION
BASE_MODEL    = 'answerdotai/ModernBERT-base'
RUN_PREFIX    = "Rank_64"
LEARNING_RATE = 5e-5
LORA_RANK     = 64
NUM_EPOCHS    = 5
BATCH_SIZE    = 32

# 3. DATA PREPARATION
df = pd.read_csv('combined03.csv')

# Fixed input format — no duplicates
df["input_text"] = (
    " | Followers: " + df["followers"].astype(str) +
    " | Year: "      + df["year"].astype(str) +
    df["text"]
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def preprocess(examples):
    return tokenizer(
        examples['input_text'],
        padding='max_length',
        truncation=True,
        max_length=256
    )

def to_log_labels(batch):
    batch["labels"] = [float(np.log1p(x)) for x in batch["likes"]]
    return batch

full_ds = Dataset.from_pandas(df[["input_text", "likes"]])
full_ds = full_ds.map(preprocess, batched=True)
full_ds = full_ds.map(to_log_labels, batched=True)
full_ds = full_ds.remove_columns(["input_text", "likes"])
full_ds = full_ds.cast_column("labels", Value("float32"))
full_ds.set_format("torch")

split      = full_ds.train_test_split(test_size=0.2, seed=SEED)
train_pool = split["train"]
test_ds    = split["test"]

print(f"Train pool: {len(train_pool)} | Test set: {len(test_ds)} (fixed)")

# 4. MODEL INITIALISER
def model_init():
    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL, num_labels=1)
    lora_config = LoraConfig(
        r=LORA_RANK,
        lora_alpha=LORA_RANK * 2,
        target_modules=["attn.Wqkv", "attn.Wo", "mlp.Wi", "mlp.Wo"],
        lora_dropout=0.1,
        bias="none",
        task_type=TaskType.SEQ_CLS
    )
    return get_peft_model(model, lora_config)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds    = logits.squeeze()
    rmse_log = np.sqrt(mean_squared_error(labels, preds))
    mae_log  = np.mean(np.abs(preds - labels))
    r2       = r2_score(labels, preds)
    return {"rmse_log": rmse_log, "mae_log": mae_log, "r2": r2}

# 5. LEARNING CURVE LOOP
SIZES = [len(train_pool)]
learning_curve_results = []

print(f"Starting learning curve across sizes: {SIZES}\n")

for size in SIZES:
    current_run_name = f"{RUN_PREFIX}_sz{size}"
    print(f"🚀 Training with dataset size: {size}")

    subset_train = train_pool.shuffle(seed=SEED).select(range(size))

    training_args = TrainingArguments(
        output_dir=f"./{current_run_name}",
        eval_strategy="epoch",
        save_strategy="epoch",
        num_train_epochs=NUM_EPOCHS,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        fp16=True,
        load_best_model_at_end=True,
        metric_for_best_model="rmse_log",
        greater_is_better=False,
        report_to="wandb",
        run_name=current_run_name,
        seed=SEED
    )

    trainer = Trainer(
        model_init=model_init,
        args=training_args,
        train_dataset=subset_train,
        eval_dataset=test_ds,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_metrics = trainer.evaluate()

    # ✅ THIS MUST BE INDENTED INSIDE THE LOOP
    learning_curve_results.append({
        "Training Size":  size,
        "Best Eval RMSE": round(eval_metrics["eval_rmse_log"], 4),
        "Best Eval MAE":  round(eval_metrics["eval_mae_log"],  4),
        "Best Eval R2":   round(eval_metrics["eval_r2"],       4),
    })

    print(f"   RMSE: {eval_metrics['eval_rmse_log']:.4f} | "
          f"MAE: {eval_metrics['eval_mae_log']:.4f} | "
          f"R²: {eval_metrics['eval_r2']:.4f}\n")

    wandb.finish()

# 6. FINAL SUMMARY TABLE
summary_df = pd.DataFrame(learning_curve_results)
print("\n" + "="*40)
print("LEARNING CURVE SUMMARY")
print(summary_df.to_string(index=False))
print("="*40)

summary_df.to_csv("learning_curve_results.csv", index=False)
print("Saved to learning_curve_results.csv")

# 7. LOG COMBINED CURVE TO W&B
summary_run = wandb.init(
    project="modernbert-engagement",
    name=f"{RUN_PREFIX}_learning_curve_summary"
)

for result in learning_curve_results:
    wandb.log({
        "training_size": result["Training Size"],
        "eval_rmse":     result["Best Eval RMSE"],
        "eval_mae":      result["Best Eval MAE"],
        "eval_r2":       result["Best Eval R2"],
    })

table = wandb.Table(
    columns=["Training Size", "Best Eval RMSE", "Best Eval MAE", "Best Eval R2"],
    data=[[r["Training Size"], r["Best Eval RMSE"],
           r["Best Eval MAE"], r["Best Eval R2"]]
          for r in learning_curve_results]
)
wandb.log({"learning_curve_table": table})
wandb.finish()

print("Learning curve summary logged to W&B.")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: shaninee (shaninee-czech-technical-university-in-prague) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/10962 [00:00<?, ? examples/s]

Map:   0%|          | 0/10962 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/10962 [00:00<?, ? examples/s]

Train pool: 8769 | Test set: 2193 (fixed)
Starting learning curve across sizes: [8769]

🚀 Training with dataset size: 8769


...

   RMSE: 2.0535 | MAE: 1.5118 | R²: 0.6519



eval/loss,█▃▂▃▁▁
eval/mae_log,█▃▂▃▁▁
eval/r2,▁▆▇▆██
eval/rmse_log,█▃▂▄▁▁
eval/runtime,█▁▂▂▂▂
eval/samples_per_second,▁█▇▇▇▇
eval/steps_per_second,▁█▇▇▇▇
train/epoch,▁▂▃▅▆▆███
train/global_step,▁▂▃▅▆▆███
train/grad_norm,█▁
+2,...



LEARNING CURVE SUMMARY
 Training Size  Best Eval RMSE  Best Eval MAE  Best Eval R2
          8769          2.0535         1.5118        0.6519
Saved to learning_curve_results.csv


eval_mae,▁
eval_r2,▁
eval_rmse,▁
training_size,▁
eval_mae,1.5118
eval_r2,0.6519
eval_rmse,2.0535
training_size,8769


Learning curve summary logged to W&B.


In [ ]:
from google.colab import drive, files

# --- 7. SAVE TO DRIVE & DOWNLOAD ---

# A. Mount Google Drive
drive.mount('/content/drive')

# Define Paths
local_model_path = f"./{RUN_NAME}"  # Where Trainer saved it temporarily
drive_save_path  = f"/content/drive/My Drive/ModernBERT_Project/{RUN_NAME}"
zip_filename     = f"{RUN_NAME}.zip"

print(f"Saving model temporarily to: {local_model_path}")
trainer.save_model(local_model_path)
tokenizer.save_pretrained(local_model_path)

# B. Copy to Google Drive
print(f"Copying to Google Drive: {drive_save_path}...")
if os.path.exists(drive_save_path):
    shutil.rmtree(drive_save_path) # Clear old version if exists
shutil.copytree(local_model_path, drive_save_path)
print("✅ Saved to Google Drive!")

# C. Download to Local Computer
print("Zipping and downloading to your computer...")
shutil.make_archive(RUN_NAME, 'zip', local_model_path) # Create zip file
files.download(zip_filename) # Trigger download
print("✅ Download started!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saving model temporarily to: ./model_w_fd_d02_h1
Copying to Google Drive: /content/drive/My Drive/ModernBERT_Project/model_w_fd_d02_h1...
✅ Saved to Google Drive!
Zipping and downloading to your computer...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started!


In [ ]:
# 8. PREVIEW RESULTS (REAL NUMBERS)
preds = trainer.predict(hf["test"]).predictions.squeeze()
labels = trainer.predict(hf["test"]).label_ids
results = pd.DataFrame({
    "True": np.expm1(labels),
    "Pred": np.maximum(0, np.expm1(preds))
})
results["Diff"] = abs(results["True"] - results["Pred"])
print(results.head(10))
print("MAE (Real):", results["Diff"].mean())

wandb.finish()

         True       Pred       Diff
0    2.000000  44.175137  42.175137
1   23.000000  86.247597  63.247597
2    0.000000  19.481689  19.481689
3   99.000008  59.908638  39.091370
4    9.000000   9.040638   0.040638
5    1.000000   3.642051   2.642051
6   15.000000  65.374474  50.374474
7    7.000000   7.621826   0.621826
8  138.000015  73.335884  64.664131
9   10.000001   9.543070   0.456931
MAE (Real): 1579.622


eval/loss,█▃▃▅▁▁
eval/mae_log,█▅▄▄▁▁
eval/r2_log,▁▆▆▄██
eval/rmse_log,█▃▃▅▁▁
eval/runtime,█▁▁▁▁▁
eval/samples_per_second,▁███▇▇
eval/steps_per_second,▁███▇▇
test/loss,▁▁
test/mae_log,▁▁
test/r2_log,▁▁
+9,...
